<a href="https://colab.research.google.com/github/Ananyasinha7/FinetuningSpectrum/blob/main/Finetuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!nvidia-smi

Sat Oct  3 19:43:48 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   56C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
!pip install transformers peft trl datasets evaluate accelerate -q

import torch
import transformers
import peft
import trl
import datasets as ds_lib

print(f"transformers:{transformers.__version__}")
print(f"peft:{peft.__version__}")
print(f"trl:{trl.__version__}")
print(f"datasets:{ds_lib.__version__}")
print(f"GPU available:{torch.cuda.is_available()}")
if torch.cuda.is_available():
  print(f"GPU name:{torch.cuda.get_device_name(0)}")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 925.5/925.5 kB 33.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 9.7 MB/s eta 0:00:00
transformers:5.17.0
peft:0.21.0
trl:1.14.1
datasets:4.8.5
GPU available:True
GPU name:Tesla T4


In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer

tokenizer=AutoTokenizer.from_pretrained("distilbert-base-uncased")

imdb=load_dataset("stanfordnlp/imdb")
small_train=imdb["train"].shuffle(seed=42).select(range(2000))
small_test=imdb["test"].shuffle(seed=42).select(range(500))

def preprocess(examples):
  return tokenizer(examples["text"], truncation=True, max_length=256)

tokenized_train=small_train.map(preprocess, batched=True)
tokenized_test=small_test.map(preprocess, batched=True)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

README.md:   0%|          | 0.00/7.81k [00:00<?, ?B/s]

...

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

In [ ]:
# setting up accuracy metric
import evaluate
accuracy_metric=evaluate.load("accuracy")
import numpy as np

# defining how to compute accuracy from model predictions
def compute_metrics(eval_pred):
  logits,labels=eval_pred #logits=raw output
  predictions=np.argmax(logits, axis=1)
  return accuracy_metric.compute(predictions=predictions, references=labels)

results={}

In [ ]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer

model_fe=AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased", num_labels=2
)


for param in model_fe.distilbert.parameters():
  param.requires_grad=False

trainable=sum(p.numel() for p in model_fe.parameters() if
              p.requires_grad)
total=sum(p.numel() for p in model_fe.parameters())
print(f"Feature Extraction - Trainable: {trainable:,}/Total:{total:,} ({100*trainable/total:.4f}%)")


model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Feature Extraction - Trainable: 592,130/Total:66,955,010 (0.8844%)


In [ ]:

training_args_fe=TrainingArguments(
    output_dir="./feat-extract",
    num_train_epochs=2,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=2e-5,
    fp16=True,
    eval_strategy="epoch",
    logging_steps=50,
    report_to="none"
)


trainer_fe=Trainer(
    model=model_fe,
    args=training_args_fe,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)

In [ ]:
trainer_fe.train()
fe_metrics=trainer_fe.evaluate()
results["Feature Extraction"]={
    "trainable": trainable,
    "total": total,
    "accuracy":fe_metrics["eval_accuracy"]
}
print(f"Feature Extraction Accuracy: {fe_metrics['eval_accuracy']:.4f}")

Epoch,Training Loss,Validation Loss,Accuracy
1,0.679667,0.670687,0.772000
2,0.661298,0.662612,0.798000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training Loss,Validation Loss,Epoch,Accuracy
0.661298,0.662612,2,0.798000


Feature Extraction Accuracy: 0.7980


In [ ]:
model_full=AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=2)

trainable_full=sum(p.numel() for p in model_full.parameters() if
                   p.requires_grad)
total_full=sum(p.numel() for p in model_full.parameters())
print(f"Full Fine-Tuning - Trainable: {trainable_full:,} / Total: {total_full:,} ({100*trainable_full/total_full:.4f}%)")

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Full Fine-Tuning - Trainable: 66,955,010 / Total: 66,955,010 (100.0000%)


In [ ]:
training_args_full=TrainingArguments(
    output_dir="./full-ft",
    num_train_epochs=2,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=2e-5,
    fp16=True,
    eval_strategy="epoch",
    logging_steps=50,
    report_to="none"
)

trainer_full=Trainer(
    model=model_full,
    args=training_args_full,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)

trainer_full.train()
full_metrics=trainer_full.evaluate()
results["Full Fine-Tuning"]={
    "trainable": trainable_full,
    "total": total_full,
    "accuracy": full_metrics["eval_accuracy"]
}
print(f"Full Fine-Tuning Accuracy: {full_metrics['eval_accuracy']:.4f}")

Epoch,Training Loss,Validation Loss,Accuracy
1,0.440917,0.325504,0.872000
2,0.243667,0.311958,0.872000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training Loss,Validation Loss,Epoch,Accuracy
0.243667,0.311958,2,0.872000


Full Fine-Tuning Accuracy: 0.8720


# LoRA

In [ ]:
!pip install -U torchao -q

from peft import LoraConfig, get_peft_model, TaskType
from transformers import AutoModelForSequenceClassification

model_lora=AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased", num_labels=2
)

lora_config=LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    target_modules=["q_lin","v_lin"],
    task_type=TaskType.SEQ_CLS
)


model_lora=get_peft_model(model_lora, lora_config)
model_lora.print_trainable_parameters()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 71.7 MB/s eta 0:00:00


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


trainable params: 739,586 || all params: 67,694,596 || trainable%: 1.0925


In [ ]:


training_args_lora=TrainingArguments(
    output_dir="./lora-ft",
    num_train_epochs=2,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=2e-4,
    fp16=True,
    eval_strategy="epoch",
    logging_steps=50,
    report_to="none"
)


trainer_lora=Trainer(
    model=model_lora,
    args=training_args_lora,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)

trainer_lora.train()
lora_metrics=trainer_lora.evaluate()



Epoch,Training Loss,Validation Loss,Accuracy
1,0.441575,0.331171,0.860000
2,0.305580,0.316565,0.864000


Training Loss,Validation Loss,Epoch,Accuracy
0.305580,0.316565,2,0.864000


In [ ]:
trainable_lora=sum(p.numel() for p in model_lora.parameters() if
                   p.requires_grad)
total_lora=sum(p.numel() for p in model_lora.parameters())
results["LoRA"]={
    "trainable": trainable_lora,
    "total": total_lora,
    "accuracy": lora_metrics["eval_accuracy"]
}
print(f"LoRA Accuracy: {lora_metrics['eval_accuracy']:.4f}")


LoRA Accuracy: 0.8640


In [ ]:
print("\n" + "=" * 65)
print(f"{'Method':<22} {'Trainable':>12} {'% of Total':>12} {'Accuracy':>10}")
print("=" * 65)
for method, data in results.items():
  pct=100* data["trainable"]/data["total"]
  print(f"{method:<22} {data['trainable']:>12,} {pct:>11.4f}% {data['accuracy']:>9.4f}")
  print("="*65)


Method                    Trainable   % of Total   Accuracy
Feature Extraction          592,130      0.8844%    0.7980
Full Fine-Tuning         66,955,010    100.0000%    0.8720
LoRA                        739,586      1.0925%    0.8640


# SFT with LoRA - Instruction Fine-Tuning a Language Model
A causal language model is different. It takes text in and generates new text, word by word. That is the type of model behind every chatbot and AI assistant.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer as AT
from peft import LoraConfig as LC
from trl import SFTTrainer, SFTConfig

sft_model_name="Qwen/Qwen3-0.6B"
sft_tokenizer=AT.from_pretrained(sft_model_name)
sft_model=AutoModelForCausalLM.from_pretrained(
    sft_model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

...

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B / 1.50GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

In [ ]:
sft_lora_config=LC(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj","v_proj"],
    task_type="CAUSAL_LM"
)

capybara=load_dataset("trl-lib/Capybara", split="train")
capybara_small=capybara.shuffle(seed=42).select(range(1000))

README.md:   0%|          | 0.00/520 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 37.2MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  482kB            

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/15806 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/200 [00:00<?, ? examples/s]

In [ ]:
sft_config = SFTConfig(
    output_dir="./sft-lora",
    num_train_epochs=1,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=10,
    max_length=512,
    report_to="none"
)

sft_trainer = SFTTrainer(
    model=sft_model,
    args=sft_config,
    train_dataset=capybara_small,
    peft_config=sft_lora_config
)

sft_trainer.train()

Tokenizing train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


Step,Training Loss
10,1.889436
20,1.650002
30,1.539334
40,1.770012
50,1.724169
60,1.650642
70,1.654464
80,1.583098
90,1.533702
100,1.551942


TrainOutput(global_step=125, training_loss=1.6347030639648437, metrics={'train_runtime': 335.9892, 'train_samples_per_second': 2.976, 'train_steps_per_second': 0.372, 'total_flos': 1306138503020544.0, 'train_loss': 1.6347030639648437, 'entropy': 1.6210837543010712, 'num_tokens': 425588.0, 'mean_token_accuracy': 0.6308203652501106, 'epoch': 1.0})

In [ ]:
sft_model.eval()

test_prompt = "Explain what LoRA fine-tuning is in simple terms."
messages = [{"role": "user", "content": test_prompt}]
input_text = sft_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = sft_tokenizer(input_text, return_tensors="pt").to(sft_model.device)

with torch.no_grad():
    outputs = sft_model.generate(
        **inputs, max_new_tokens=150, do_sample=True, temperature=0.7
    )


response = sft_tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)
print(f"\nPrompt: {test_prompt}")
print(f"Response: {response}")


Prompt: Explain what LoRA fine-tuning is in simple terms.
Response: LoRA is a type of fine-tuning technique used to improve the performance of a model when it's trained on a large dataset. The idea is that the model already knows how to perform a certain task, but it might not be able to do it as well on a different dataset. LoRA fine-tuning works by adding a small amount of new information that helps the model perform the task better on its own. This is done by modifying the original model's weights in a way that makes it more effective at performing the task.

LoRA is commonly used in large language models and other machine learning models. It's not a complete fine-tuning process, but it's a way to give the model more information about how to perform the task.


In [ ]:
print("\n" + "=" * 90)
print("FINE-TUNING SPECTRUM SUMMARY")
print("=" * 90)
print(f"{'Method':<25} {'What is Trained':<25} {'Params %':<12} {'Use Case':<28}")
print("-" * 90)
print(f"{'Feature Extraction':<25} {'Classification head only':<25} {'<0.01%':<12} {'Quick baseline, limited data':<28}")
print(f"{'Full Fine-Tuning':<25} {'All model parameters':<25} {'100%':<12} {'Max accuracy, small models':<28}")
print(f"{'LoRA (PEFT)':<25} {'Low-rank adapter matrices':<25} {'<1%':<12} {'Production fine-tuning':<28}")
print(f"{'SFT + LoRA':<25} {'LoRA on causal LM':<25} {'<1%':<12} {'Instruction/chat tuning':<28}")
print(f"{'Instruction FT':<25} {'= SFT with instruction data':<25} {'varies':<12} {'Chat models, assistants':<28}")
print("=" * 90)
print("\nKey insight: SFT and Instruction Fine-Tuning are training METHODOLOGIES.")
print("Feature Extraction, Full FT, and LoRA are STRATEGIES for which parameters to update.")
print("You can combine them: e.g., SFT + LoRA = instruction-tune with parameter efficiency.")


FINE-TUNING SPECTRUM SUMMARY
Method                    What is Trained           Params %     Use Case                    
------------------------------------------------------------------------------------------
Feature Extraction        Classification head only  <0.01%       Quick baseline, limited data
Full Fine-Tuning          All model parameters      100%         Max accuracy, small models  
LoRA (PEFT)               Low-rank adapter matrices <1%          Production fine-tuning      
SFT + LoRA                LoRA on causal LM         <1%          Instruction/chat tuning     
Instruction FT            = SFT with instruction data varies       Chat models, assistants     

Key insight: SFT and Instruction Fine-Tuning are training METHODOLOGIES.
Feature Extraction, Full FT, and LoRA are STRATEGIES for which parameters to update.
You can combine them: e.g., SFT + LoRA = instruction-tune with parameter efficiency.


# QLoRA


In [ ]:
!pip install bitsandbytes -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 25.9 MB/s eta 0:00:00


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig
from datasets import load_dataset

qlora_model_name="Qwen/Qwen3-0.6B"

# Configure 4-bit quantization with NormalFloat4 format
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

qlora_tokenizer = AutoTokenizer.from_pretrained(qlora_model_name)
qlora_model = AutoModelForCausalLM.from_pretrained(
    qlora_model_name,
    quantization_config=bnb_config,
    device_map="auto",
)

mem_bytes = qlora_model.get_memory_footprint()
print(f"QLoRA model memory footprint: {mem_bytes / 1e9:.2f} GB")
print(f"(Compare to ~1.2 GB for float16 version)")

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

QLoRA model memory footprint: 0.53 GB
(Compare to ~1.2 GB for float16 version)


In [ ]:
qlora_lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],
    task_type="CAUSAL_LM",
)

capybara = load_dataset("trl-lib/Capybara", split="train")
capybara_small = capybara.shuffle(seed=42).select(range(500))

In [ ]:
qlora_sft_config = SFTConfig(
    output_dir="./qlora-sft",
    num_train_epochs=1,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=10,
    max_length=512,
    report_to="none",
)

qlora_trainer = SFTTrainer(
    model=qlora_model,
    args=qlora_sft_config,
    train_dataset=capybara_small,
    peft_config=qlora_lora_config,
)

qlora_trainer.train()

Tokenizing train dataset:   0%|          | 0/500 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/500 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/500 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/500 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


NotImplementedError: "_amp_foreach_non_finite_check_and_unscale_cuda" not implemented for 'BFloat16'

In [ ]:
qlora_model.eval()
test_prompt = "What is the difference between LoRA and QLoRA?"
messages = [{"role": "user", "content": test_prompt}]
input_text = qlora_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = qlora_tokenizer(input_text, return_tensors="pt").to(qlora_model.device)

with torch.no_grad():
    outputs = qlora_model.generate(
        **inputs, max_new_tokens=150, do_sample=True, temperature=0.7
    )

response = qlora_tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)
print(f"\nPrompt: {test_prompt}")
print(f"Response: {response}")


Prompt: What is the difference between LoRA and QLoRA?
Response: <think>
Okay, the user is asking about the difference between LoRA and QLoRA. I need to recall what these are. LoRA is a method used in large language models for training a new model on a large dataset. It's similar to how you train on a large dataset for a specific task. But QLoRA seems to be a variation, maybe a different implementation or a more specific approach?

I remember that QLoRA is a variant of LoRA, but there's a difference. QLoRA might be a more efficient version, using a different loss function, like a different loss function than the standard LoRA. Or maybe it's for a different task, like a specific dataset. 

Wait, I think QLo


In [ ]:
print("\n" + "=" * 90)
print("COMPLETE FINE-TUNING SPECTRUM (with QLoRA)")
print("=" * 90)
print(f"{'Method':<25} {'What is Trained':<25} {'Params %':<12} {'Use Case':<28}")
print("-" * 90)
print(f"{'Feature Extraction':<25} {'Classification head only':<25} {'<0.01%':<12} {'Quick baseline, limited data':<28}")
print(f"{'Full Fine-Tuning':<25} {'All model parameters':<25} {'100%':<12} {'Max accuracy, small models':<28}")
print(f"{'LoRA (PEFT)':<25} {'Low-rank adapter matrices':<25} {'<1%':<12} {'Production fine-tuning':<28}")
print(f"{'SFT + LoRA':<25} {'LoRA on causal LM':<25} {'<1%':<12} {'Instruction/chat tuning':<28}")
print(f"{'QLoRA':<25} {'LoRA + 4-bit base model':<25} {'<1%':<12} {'Large models, limited VRAM':<28}")
print("=" * 90)


COMPLETE FINE-TUNING SPECTRUM (with QLoRA)
Method                    What is Trained           Params %     Use Case                    
------------------------------------------------------------------------------------------
Feature Extraction        Classification head only  <0.01%       Quick baseline, limited data
Full Fine-Tuning          All model parameters      100%         Max accuracy, small models  
LoRA (PEFT)               Low-rank adapter matrices <1%          Production fine-tuning      
SFT + LoRA                LoRA on causal LM         <1%          Instruction/chat tuning     
QLoRA                     LoRA + 4-bit base model   <1%          Large models, limited VRAM  
